# Import all the libraries for extracting and appending page by page, then create the data frame.
- re - is a reguler expresion, which is matching with te expresion that would be extracted.
- request - is a fetching the data through url E-Commerce website.
- BeautifulSoup - is a fetched data is convert to the searchable formate.
- numpy - is a mathamatical perpose, mainly used, if column or row is empty that fill with an NaN.
- pandas - pandas for Data Manipulation

In [1]:
import re
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd

In [28]:
BRAND = []
PRICE = []
DOORS = []
WATER_CAPACITY = []
STARS = [] 
WARRANTY = []
STABILIZER = []
COMPRESSOR_MODEL = []
COMPRESSOR_WARRANTY = []
PRICE_OFF = []
RATINGS = []

PRODUCTS_PER_PAGE = 24

for page in range(1, 31):
    url = f"https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&as-pos=1&as-type=HISTORY&page={page}"
    print(url)

    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")
    
    # PRODUCT NAMES
    data = soup.find_all('div', class_="RG5Slk")

    assert len(data) >= PRODUCTS_PER_PAGE, f"Only {len(data)} products found on page {page}"

    data = data[:PRODUCTS_PER_PAGE]
    for i in data:
        BRAND.append(i.text.split()[0])

    # WATER_CAPACITY
    for i in data:
        a = re.findall(r"\d{1,3}\sL",i.text,flags=re.I)
        if len(a) > 0:
            WATER_CAPACITY.append(a[0])
        else:
            WATER_CAPACITY.append(np.nan)
        
    # PRICE
    cost = soup.find_all('div', class_="hZ3P6w DeU9vF")[:PRODUCTS_PER_PAGE]
    for i in cost:
        PRICE.append(i.text.strip())

    # STARS
    data = soup.find_all('div', class_="RG5Slk")
    s = data[:PRODUCTS_PER_PAGE]

    for i in s:
        a = re.findall(r"\d\s*star", i.text, flags=re.I)
        STARS.append(a[0] if a else np.nan)
    
    # DOORS 
    for i in data:
        v = re.findall(r"Single\sDoor|Double\sDoor", i.text, flags=re.I)
        DOORS.append(v[0] if v else np.nan)


    # SPECS (ASSERT + SLICE)
    p_data = soup.find_all('li', class_="DTBslk")[:PRODUCTS_PER_PAGE * 3]

    
    # COMPRESSOR / STABILIZER / WARRANTY
    for i in range(0, len(p_data), 3):

        specs = []   #temporary holder for one product
    
        # instead of direct indexing safe checks
        if i < len(p_data):
            specs.append(p_data[i].text.strip())
        if i+1 < len(p_data):
            specs.append(p_data[i+1].text.strip())
        if i+2 < len(p_data):
            specs.append(p_data[i+2].text.strip())
    
        #fill missing specs
        while len(specs) < 3:
            specs.append(np.nan)
    
        comp = stab = np.nan
        product_warranty = np.nan
        compressor_warranty = np.nan
        
        for s in specs:
            s_low = str(s).lower()

            if ("compressor" in s_low and "year" not in s_low and "warranty" not in s_low 
                and "product" not in s_low and "unit" not in s_low and comp is np.nan):
                comp = s

            if "stabilizer" in s_low and stab is np.nan:
                stab = s
            
        COMPRESSOR_MODEL.append(comp)
        STABILIZER.append(stab)


        warranty_text = np.nan  

        for s in specs:
            if isinstance(s, str) and "warranty" in s.lower():
                warranty_text = s
                break
        
            # Warranty of product
        w = re.findall(r"\d+\s+year\s+.*?warranty", str(warranty_text), flags=re.I)
        WARRANTY.append(w[0] if w else np.nan)

        # Compressor Warrenty
        years = re.findall(r"\d{1,2}\s+year[s]?", str(warranty_text), flags=re.I)
        COMPRESSOR_WARRANTY.append(years[1] if len(years) > 1 else np.nan)



    # RATINGS
    rtg = soup.find_all('div', class_="MKiFS6")[:PRODUCTS_PER_PAGE]

    for i in rtg:
        RATINGS.append(i.text.strip() if i.text.strip() else np.nan)

    
    # OFFERS
    off = soup.find_all('div', class_="HQe8jr")[:PRODUCTS_PER_PAGE]

    for i in off:
        PRICE_OFF.append(i.text.strip() if i.text.strip() else np.nan)


https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&as-pos=1&as-type=HISTORY&page=1
https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&as-pos=1&as-type=HISTORY&page=2
https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&as-pos=1&as-type=HISTORY&page=3
https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&as-pos=1&as-type=HISTORY&page=4
https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&as-pos=1&as-type=HISTORY&page=5
https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&as-pos=1&as-type=HISTORY&page=6
https://www.flipkart.com/search?q=refrigerator&otracker=search&otracker1=search&marketplace=FL

In [ ]:
# Here by chance if minimum length of the any column then the same length should be taken remaining columns

In [29]:
print("BRAND",len(BRAND)),
print("PRICE", len(PRICE)),
print("DOORS", len(DOORS)),
print("WATER_CAPACITY", len(WATER_CAPACITY)),
print("STARS",len(STARS)),
print("WARRANTY",len(WARRANTY)),
print("STABILIZER",len(STABILIZER)),
print("COMPRESSOR_MODEL",len(COMPRESSOR_MODEL)),
print("COMPRESSOR_WARRANTY",len(COMPRESSOR_WARRANTY))
print("PRICE_OFF",len(PRICE_OFF))
print("RATINGS",len(RATINGS))

BRAND 720
PRICE 720
DOORS 720
WATER_CAPACITY 720
STARS 720
WARRANTY 719
STABILIZER 719
COMPRESSOR_MODEL 719
COMPRESSOR_WARRANTY 719
PRICE_OFF 720
RATINGS 720


In [30]:
WARRANTY

['1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 year comprehensive warranty',
 '1 Year Warranty',
 '1 year comprehensive warranty',
 '1 Year Comprehensive Warranty',
 nan,
 '1 year comprehensive warranty',
 '1 Year Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 year comprehensive warranty',
 '1 Year Warranty',
 '1 Year Comprehensive Warranty',
 nan,
 nan,
 '1 year comprehensive warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 year comprehensive warranty',
 '1 Year Comprehensive Warranty',
 nan,
 '1 Year Comprehensive warranty',
 '1 Year Comprehensive Warranty',
 '1 year com

In [31]:
min_len = min(
    len(BRAND),
    len(PRICE),
    len(DOORS),
    len(WATER_CAPACITY),
    len(STARS),
    len(WARRANTY),
    len(STABILIZER),
    len(COMPRESSOR_MODEL),
    len(COMPRESSOR_WARRANTY),
    len(PRICE_OFF),
    len(RATINGS)
)

In [32]:
df = pd.DataFrame({
    "BRAND": BRAND[:min_len],
    "PRICE": PRICE[:min_len],
    "DOORS": DOORS[:min_len],
    "WATER_CAPACITY": WATER_CAPACITY[:min_len],
    "STARS": STARS[:min_len],
    "WARRANTY": WARRANTY[:min_len],
    "STABILIZER": STABILIZER[:min_len],
    "COMPRESSOR_MODEL": COMPRESSOR_MODEL[:min_len],
    "COMPRESSOR_WARRANTY": COMPRESSOR_WARRANTY[:min_len],
    "PRICE_OFF": PRICE_OFF[:min_len],
    "RATINGS": RATINGS[:min_len]
})

In [33]:
df

,BRAND,PRICE,DOORS,WATER_CAPACITY,STARS,WARRANTY,STABILIZER,COMPRESSOR_MODEL,COMPRESSOR_WARRANTY,PRICE_OFF,RATINGS
0,Whirlpool,"₹15,790",Single Door,192 L,4 Star,1 Year Comprehensive Warranty,Built-in Stabilizer,Inverter Compressor,9 Years,28% off,4.4
1,Whirlpool,"₹16,790",Single Door,184 L,5 Star,1 Year Comprehensive Warranty,Built-in Stabilizer,Inverter Compressor,9 Years,22% off,4.4
2,Samsung,"₹13,990",Single Door,183 L,2 Star,1 Year Comprehensive Warranty,Built-in Stabilizer,NaN,10 Years,30% off,4.5
3,Samsung,"₹16,990",Single Door,183 L,5 Star,1 year comprehensive warranty,Built-in Stabilizer,NaN,20 years,26% off,4.5
4,Haier,"₹11,990",Single Door,185 L,2 Star,1 Year Warranty,Built-in Stabilizer,Normal Compressor,10 Years,29% off,4.3
...,...,...,...,...,...,...,...,...,...,...,...
714,Lloyd,"₹22,310",Double Door,240 L,2 Star,1 Year Warranty,Built-in Stabilizer,Reciprocatory Compressor,10 Years,46% off,4.3
715,Whirlpool,"₹26,990",Double Door,260 L,2 Star,1 Year Comprehensive Warranty,Built-in Stabilizer,Inverter Compressor,10 Years,25% off,4.2
716,Godrej,"₹24,990",Double Door,244 L,4 Star,NaN,Built-in Stabilizer,Digital Inverter Compressor,NaN,37% off,4.5
717,Haier,"₹60,990",NaN,602 L,NaN,1 year comprehensive warranty,Built-in Stabilizer,Normal Compressor,20 year,44% off,4.1


In [34]:
df.WARRANTY.unique()

array(['1 Year Comprehensive Warranty', '1 year comprehensive warranty',
       '1 Year Warranty', nan, '1 Year Comprehensive warranty',
       '1 Year warranty', '1 year comprehensive product warranty',
       '2 Year Warranty',
       '1 year complete, 4 year on compressor and 9 year on internal leakage, Warranty',
       '1 Year full warranty', '1  Year Warranty',
       '1 Year on Product Warranty', '10 Year Warranty',
       '2 YEAR COMPLETE AND 10 YEARS COMPRESSOR WARRANTY',
       '3 Year Comprehensive Warranty', '1 year Warranty',
       '1 Year Product Warranty', '1 year product warranty'], dtype=object)

In [35]:
COMPRESSOR_WARRANTY

['9 Years',
 '9 Years',
 '10 Years',
 '20 years',
 '10 Years',
 '20 years',
 '10 Years',
 nan,
 '20 years',
 '10 Years',
 '9 Years',
 '10 Years',
 '10 Years',
 '10 Years',
 '10 Years',
 '10 Years',
 '10 Years',
 '9 Years',
 '20 years',
 '10 Years',
 '10 Years',
 nan,
 nan,
 '20 years',
 '10 Years',
 '10 Years',
 '9 Years',
 '10 Years',
 '10 Years',
 '20 year',
 '9 Years',
 nan,
 '10 Years',
 '10 Years',
 '20 year',
 '10 Year',
 '9 Years',
 '9 Years',
 '20 years',
 '10 Years',
 '10 Years',
 '10 Years',
 '10 Year',
 '10 Years',
 '10 Years',
 '10 Years',
 '9 Years',
 nan,
 '10 Year',
 '10 Years',
 '9 Years',
 '10 Years',
 '10 years',
 '10 Years',
 '9 Years',
 '10 Years',
 '9 Years',
 nan,
 '10 Years',
 nan,
 '9 Years',
 '9 Years',
 '10 Years',
 '10 Years',
 '9 Years',
 '20 years',
 '10 Years',
 '10 Years',
 '9 Years',
 '10 Years',
 '9 Years',
 nan,
 '10 Years',
 '12 Years',
 '10 Years',
 '10 Years',
 '9 Years',
 '9 Years',
 '10 Years',
 '10 Years',
 '10 Years',
 nan,
 '20 years',
 '10 Yea

In [36]:
df.STABILIZER.unique()

array(['Built-in Stabilizer', nan], dtype=object)

In [37]:
COMPRESSOR_MODEL

['Inverter Compressor',
 'Inverter Compressor',
 nan,
 nan,
 'Normal Compressor',
 nan,
 'Reciprocating Compressor',
 'Normal Compressor',
 'Digital Inverter Compressor',
 'Normal Compressor',
 'Normal Compressor',
 'Inverter Compressor',
 'Reciprocating Compressor',
 'Inverter Compressor',
 'Reciprocating Compressor',
 'Reciprocating Compressor',
 'Reciprocatory Compressor',
 'Inverter Compressor',
 'Digital Inverter Compressor',
 'Reciprocating Compressor',
 'Reciprocatory Compressor',
 'Inverter Compressor',
 'Normal Compressor',
 nan,
 'Reciprocating Compressor',
 'Inverter Compressor',
 'Inverter Compressor',
 'Reciprocatory Compressor',
 'Reciprocating Compressor',
 nan,
 'Inverter Compressor',
 'Normal Compressor',
 'Pro Smart Inverter Compressor',
 'Smart Inverter Compressor',
 nan,
 'Pro Smart Inverter Compressor',
 'Inverter Compressor',
 'Normal Compressor',
 nan,
 'Inverter Compressor',
 'Inverter Compressor',
 'Reciprocatory Compressor',
 'Smart Inverter Compressor',
 'Rec

In [38]:
df.shape

(719, 11)

In [39]:
df.isna().sum()

BRAND                    0
PRICE                    0
DOORS                   68
WATER_CAPACITY           0
STARS                   52
WARRANTY               175
STABILIZER              19
COMPRESSOR_MODEL        79
COMPRESSOR_WARRANTY    141
PRICE_OFF                0
RATINGS                  0
dtype: int64

# Save with CSV file ?

In [40]:
df.to_csv("Web_Scraping_Refrigerators_data_df3.csv")

In [23]:
specs

['Inverter Compressor',
 'Built-in Stabilizer',
 '1 Year Domestic Warranty on Product and Additional 9 Years on Compressor']

In [24]:
STABILIZER

['Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabilizer',
 'Built-in Stabi

In [25]:
COMPRESSOR_MODEL

['Reciprocating Compressor',
 'Reciprocatory Compressor',
 nan,
 nan,
 'Reciprocating Compressor',
 'Normal Compressor',
 'Smart Inverter Compressor',
 'Reciprocatory Compressor',
 'Normal Compressor',
 'Digital Inverter Compressor',
 'Reciprocatory Compressor',
 nan,
 'Digital Inverter Compressor',
 'Smart inverter compressor',
 'Reciprocating Compressor',
 'Reciprocating Compressor',
 'Reciprocatory Compressor',
 'Reciprocatory Compressor',
 'Inverter Compressor',
 nan,
 'Reciprocating Compressor',
 'Digital Inverter Compressor',
 'Reciprocating Compressor',
 'Reciprocating Compressor',
 'Inverter Compressor',
 'Reciprocating Compressor',
 'Reciprocatory Compressor',
 'Reciprocating Compressor',
 'Inverter Compressor',
 'Inverter Compressor',
 'Reciprocatory Compressor',
 'Smart inverter compressor',
 'Digital Inverter Compressor',
 nan,
 nan,
 'Inverter Compressor',
 'Reciprocatory Compressor',
 nan,
 'Inverter Compressor',
 'Reciprocating Compressor',
 'Reciprocatory Compressor',
 

In [21]:
WARRANTY

['1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Warranty',
 '1 year comprehensive warranty',
 '1 year comprehensive warranty',
 '1 year comprehensive warranty',
 nan,
 '1 Year Comprehensive Warranty',
 '1 Year Warranty',
 nan,
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 year comprehensive warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Warranty',
 '1 year comprehensive warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Product Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 '1 Year Comprehensive Warranty',
 nan,
 nan,
 '1 Year Comprehensive Warranty',
 '1 year comprehensive warranty',
 '1 year comprehensive warranty'